# P04B — Eksplorasi dan pemodelan algoritma

Pendamping Bab 4. Jalankan P04A terlebih dahulu pada folder kerja yang sama (di Colab, sesi yang sama; di komputer lokal, jalankan dari `praktikum/bab04/`). Notebook ini memperlihatkan bagaimana beberapa keluarga algoritma digunakan pada data sintesis Bab 4: klasifikasi (subbab 4.2), baseline, pipeline, dan perbandingan regresi (4.4–4.5), clustering dan PCA (4.3.1, 4.3.3), serta anomali (4.3.2). Hasil latihan tidak menggambarkan kinerja model lapangan. Evaluasi metrik dan tuning sistematis diperdalam pada Bab 5–6.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, IsolationForest
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Mencari folder data: folder kerja (hasil P04A) atau jalur dari akar repositori.
kandidat = [Path("data_bab4"), Path("praktikum/bab04/data_bab4")]
DATA = next((p for p in kandidat if (p / "data_sintesis_kuat_tekan.csv").exists()), None)
if DATA is None:
    raise FileNotFoundError("Folder data_bab4 tidak ditemukan. Jalankan P04A terlebih dahulu pada folder kerja yang sama.")
print("Folder data:", DATA)

Folder data: data_bab4


## 1. Periksa skema dan sasaran
Pastikan identitas dan label simulasi tidak masuk sebagai prediktor. Pada contoh jembatan, label memang dibuat dengan aturan sintesis sehingga pemodelan hanya mengajarkan alur supervised learning.

In [2]:
beton = pd.read_csv(DATA / "data_sintesis_kuat_tekan.csv")
jembatan = pd.read_csv(DATA / "data_sintesis_prioritas_jembatan.csv")
perkerasan = pd.read_csv(DATA / "data_sintesis_perkerasan.csv")
sensor = pd.read_csv(DATA / "data_sintesis_sensor.csv")
for nama, df in [("beton", beton),("jembatan",jembatan),("perkerasan",perkerasan),("sensor",sensor)]:
    print(nama, df.shape, "missing:", int(df.isna().sum().sum()))
print("Kelas label sintesis jembatan:")
print(jembatan["prioritas_sintesis"].value_counts().to_dict())


beton (1200, 10) missing: 0
jembatan (600, 7) missing: 0
perkerasan (800, 6) missing: 0
sensor (700, 4) missing: 0
Kelas label sintesis jembatan:
{'prioritas': 308, 'rutin': 292}


## 2. Klasifikasi label latihan jembatan
Label sintesis dibuat dari ambang pada skor simulasi yang menggabungkan lima prediktor. Model mempelajari label tersebut; label bukan kategori inspeksi resmi. ID aset dikeluarkan dari fitur.


In [3]:
bridge_features = ["umur_tahun", "lalin_berat_harian", "risiko_gerusan", "indeks_deteriorasi", "paparan_banjir"]
Xb = jembatan[bridge_features]; yb = jembatan["prioritas_sintesis"]
Xb_train, Xb_test, yb_train, yb_test = train_test_split(Xb, yb, test_size=0.25, random_state=41804, stratify=yb)
classifiers = {"naive_bayes": GaussianNB(),
               "pohon": DecisionTreeClassifier(max_depth=4, min_samples_leaf=8, random_state=41804),
               "regresi_logistik": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))}
for name, clf in classifiers.items():
    clf.fit(Xb_train, yb_train)
    print(name, "contoh kelas:", clf.predict(Xb_test.iloc[:5]).tolist())

# Keluaran model adalah peluang; kelas muncul setelah ambang dipilih (subbab 4.2.1).
logit = classifiers["regresi_logistik"]
p_prioritas = logit.predict_proba(Xb_test)[:, list(logit.classes_).index("prioritas")]
print("\npeluang prioritas, lima contoh uji:", p_prioritas[:5].round(2).tolist())
for ambang in (0.3, 0.5, 0.7):
    print(f"ambang {ambang}: {int((p_prioritas >= ambang).sum())} dari {len(p_prioritas)} data uji ditandai prioritas")

naive_bayes contoh kelas: ['prioritas', 'prioritas', 'rutin', 'prioritas', 'prioritas']
pohon contoh kelas: ['prioritas', 'prioritas', 'rutin', 'prioritas', 'prioritas']
regresi_logistik contoh kelas: ['prioritas', 'prioritas', 'rutin', 'prioritas', 'prioritas']

peluang prioritas, lima contoh uji: [1.0, 0.98, 0.0, 0.89, 0.97]
ambang 0.3: 85 dari 150 data uji ditandai prioritas
ambang 0.5: 74 dari 150 data uji ditandai prioritas
ambang 0.7: 67 dari 150 data uji ditandai prioritas


## 3. Baseline, pipeline, dan perbandingan regresi
Semua kandidat menerima fitur dan pembagian pelatihan yang sama (subbab 4.4–4.5). **Baseline** (rerata target pada data latih) memberi pembanding: model yang tidak mengalahkannya belum menambah manfaat. **Pipeline** menggabungkan penskalaan dan estimator agar penskalaan hanya dipelajari dari data latih. Perbandingan satu pembagian belum cukup untuk kesimpulan umum, jadi bagian berikutnya mengulang pembagian dengan seed berbeda. Tabel ini bukan peringkat metode dan tidak menggantikan evaluasi pada Bab 5.

In [4]:
features = [c for c in beton.columns if c not in ["id_spesimen", "kuat_tekan_MPa"]]
X = beton[features]
y = beton["kuat_tekan_MPa"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=41804)

def buat_model():
    return {"baseline_rerata": DummyRegressor(strategy="mean"),
            "linear": LinearRegression(),
            "pohon": DecisionTreeRegressor(max_depth=5, random_state=41804),
            "random_forest": RandomForestRegressor(n_estimators=100, min_samples_leaf=3, random_state=41804, n_jobs=-1),
            "gradient_boosting": GradientBoostingRegressor(random_state=41804),
            "knn_tanpa_skala": KNeighborsRegressor(n_neighbors=7),
            "knn_pipeline": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=7))}

baris = []
for name, model in buat_model().items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    baris.append([name, mean_absolute_error(y_test, pred), root_mean_squared_error(y_test, pred), float(np.mean(y_test - pred))])
    print(name, "contoh prediksi:", pred[:5].round(2).tolist())
print("\nSatu pembagian (seed 41804); residual = aktual - prediksi, rerata residual mendekati 0 berarti tidak bias ke satu arah")
print(pd.DataFrame(baris, columns=["model", "MAE_MPa", "RMSE_MPa", "rerata_residual_MPa"]).round(2).to_string(index=False))

baseline_rerata contoh prediksi: [32.8, 32.8, 32.8, 32.8, 32.8]
linear contoh prediksi: [31.98, 23.64, 37.55, 37.25, 31.41]
pohon contoh prediksi: [35.5, 22.1, 32.06, 41.16, 28.54]


random_forest contoh prediksi: [34.57, 25.45, 35.43, 39.74, 28.24]
gradient_boosting contoh prediksi: [32.76, 25.79, 37.59, 38.52, 29.35]
knn_tanpa_skala contoh prediksi: [28.76, 24.88, 32.07, 35.69, 35.88]
knn_pipeline contoh prediksi: [30.77, 24.46, 36.93, 38.7, 29.91]

Satu pembagian (seed 41804); residual = aktual - prediksi, rerata residual mendekati 0 berarti tidak bias ke satu arah
            model  MAE_MPa  RMSE_MPa  rerata_residual_MPa
  baseline_rerata     6.89      8.44                -0.50
           linear     3.10      3.86                 0.17
            pohon     3.42      4.19                 0.01
    random_forest     2.94      3.60                -0.09
gradient_boosting     2.70      3.35                -0.08
  knn_tanpa_skala     4.85      5.95                 0.33
     knn_pipeline     3.73      4.62                 0.02


### Satu pembagian belum cukup
Ulangi pembagian latih-uji dengan lima seed dan lihat sebaran RMSE. Bila selisih antarmodel lebih kecil daripada sebaran antarpembagian, urutan model pada satu pembagian tidak dapat dijadikan kesimpulan.

In [5]:
hasil = {n: [] for n in buat_model()}
for seed in range(5):
    Xa, Xu, ya, yu = train_test_split(X, y, test_size=0.25, random_state=seed)
    for name, model in buat_model().items():
        hasil[name].append(root_mean_squared_error(yu, model.fit(Xa, ya).predict(Xu)))
ringkas = pd.DataFrame({n: [np.mean(v), np.std(v, ddof=1), np.min(v), np.max(v)] for n, v in hasil.items()},
                       index=["rerata_RMSE", "sd_RMSE", "min", "maks"]).T
print("RMSE (MPa) pada 5 pembagian acak berbeda")
print(ringkas.round(2).to_string())

RMSE (MPa) pada 5 pembagian acak berbeda
                   rerata_RMSE  sd_RMSE   min  maks
baseline_rerata           8.80     0.08  8.70  8.88
linear                    3.94     0.09  3.85  4.06
pohon                     4.44     0.15  4.22  4.59
random_forest             3.64     0.08  3.50  3.70
gradient_boosting         3.41     0.05  3.36  3.48
knn_tanpa_skala           5.94     0.25  5.64  6.31
knn_pipeline              4.70     0.10  4.64  4.88


**Cara membaca**: bandingkan `knn_tanpa_skala` dengan `knn_pipeline`. Fitur beton berkisar dari sekitar 0,3 (rasio) hingga lebih dari 1.000 (agregat, kg/m³), sehingga jarak mentah didominasi fitur berskala besar (subbab 4.2.4). Perhatikan juga jarak antara model-model lain terhadap baseline.

## 4. Clustering dan PCA
Buang ID dan label pembangkit sebelum membentuk kelompok. Standardisasi dilakukan sebelum menghitung jarak. Komponen PCA merangkum variasi, bukan peringkat kepentingan rekayasa.

In [6]:
cols = ["retak_persen", "kedalaman_alur_mm", "IRI_m_km", "lalu_lintas_tahunan"]
Z = StandardScaler().fit_transform(perkerasan[cols])
clusters = KMeans(n_clusters=4, n_init=10, random_state=41804).fit_predict(Z)
coords = PCA(n_components=2, random_state=41804).fit_transform(Z)
print("Jumlah anggota tiap cluster:", pd.Series(clusters).value_counts().sort_index().to_dict())
print("PCA dua komponen; proporsi variasi:", PCA(n_components=2).fit(Z).explained_variance_ratio_.round(3).tolist())

# Dua metode clustering pada fitur yang sama (Latihan 5). Label pembangkit hanya dipakai SESUDAH clustering.
hierarkis = AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(Z)
for nama, lab in [("k-means", clusters), ("hierarkis (Ward)", hierarkis)]:
    print(f"{nama}: silhouette {silhouette_score(Z, lab):.3f}; ARI terhadap label pembangkit {adjusted_rand_score(perkerasan['label_pembangkit'], lab):.3f}")
print("Kesepakatan k-means vs hierarkis (ARI):", round(adjusted_rand_score(clusters, hierarkis), 3))

Jumlah anggota tiap cluster: {0: 198, 1: 200, 2: 201, 3: 201}
PCA dua komponen; proporsi variasi: [0.574, 0.318]
k-means: silhouette 0.603; ARI terhadap label pembangkit 0.990
hierarkis (Ward): silhouette 0.603; ARI terhadap label pembangkit 1.000
Kesepakatan k-means vs hierarkis (ARI): 0.99


**Cara membaca**: kesepakatan tinggi dengan label pembangkit terjadi karena data dibangkitkan dari empat gugus yang terpisah jelas; pada data nyata hal ini jarang terjadi. Silhouette menilai pemisahan menurut fitur dan jarak yang dipilih, bukan makna rekayasa kelompok. Uji kestabilan dengan mengganti seed, fitur, dan skala sebelum memberi nama kelompok.

## 5. Menyaring anomali untuk pemeriksaan
Model anomali tidak mendiagnosis kerusakan. Bandingkan baris bertanda oleh generator dengan hasil Isolation Forest hanya sebagai latihan, lalu telaah konteks dan catatan sensor.

In [7]:
sensor["flag_model"] = IsolationForest(contamination=0.03, random_state=41804).fit_predict(sensor[["suhu_C", "getaran_mm_s"]]) == -1
print("Flag generator:", int(sensor["flag_pembangkit"].sum()))
print("Flag Isolation Forest:", int(sensor["flag_model"].sum()))
print("Ditandai keduanya:", int((sensor["flag_model"] & sensor["flag_pembangkit"]).sum()),
      "| ditandai model tetapi bukan flag generator:", int((sensor["flag_model"] & ~sensor["flag_pembangkit"]).sum()),
      "| flag generator yang terlewat:", int((~sensor["flag_model"] & sensor["flag_pembangkit"]).sum()))
print(sensor.loc[sensor["flag_model"], ["waktu", "suhu_C", "getaran_mm_s", "flag_pembangkit"]].head(8).to_string(index=False))

Flag generator: 13
Flag Isolation Forest: 21
Ditandai keduanya: 13 | ditandai model tetapi bukan flag generator: 8 | flag generator yang terlewat: 0
              waktu  suhu_C  getaran_mm_s  flag_pembangkit
2024-01-01 21:00:00   16.29         1.521             True
2024-01-04 01:00:00   21.53         1.929             True
2024-01-05 19:00:00   14.31         0.425            False
2024-01-08 06:00:00   25.48         0.712            False
2024-01-08 10:00:00   22.58         1.628             True
2024-01-09 01:00:00   21.09         1.684             True
2024-01-10 22:00:00   17.93         1.849             True
2024-01-12 04:00:00   24.44         1.306             True


**Cara membaca**: pada data sintesis ini semua penyimpangan yang ditanam generator tertangkap, tetapi model juga menandai baris lain, termasuk nilai pada jam dengan suhu tinggi atau rendah yang normal menurut siklus harian. Model ini tidak memakai urutan waktu, sehingga kondisi normal dapat tertandai (subbab 4.3.2). Tandai untuk diperiksa, bukan untuk didiagnosis.

## 6. Batas penggunaan
Semua hasil bergantung pada seed, rentang dan aturan generator. Jangan mengutip hasil ini sebagai kinerja terhadap beton, jembatan, jalan atau sensor nyata. Simpan pemilihan fitur, target, seed dan pembagian data agar latihan dapat diulang.